# 📈 DRW Crypto Market Prediction | Time Series + Ensemble

Welcome to my solution for the **DRW Crypto Market Prediction** Kaggle competition!

---

> _If this helps you, consider giving it an upvote ❤️_


<a id='Imports'></a>
# Imports

In [1]:
import sys
import pandas as pd
import shap
import numpy as np
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from scipy.stats import pearsonr
from quantile_forest import RandomForestQuantileRegressor

from catboost import CatBoostRegressor, Pool

/Users/Sigrid/Desktop/drw-crypto-market-prediction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<a id='feature'></a>
# Feature Engineering

In [58]:
def feature_engineering(df):
    df['volume_weighted_sell'] = df['sell_qty'] * df['volume']
    df['buy_sell_ratio'] = df['buy_qty'] / (df['sell_qty'] + 1e-8)
    df['selling_pressure'] = df['sell_qty'] / (df['volume'] + 1e-8)
    df['effective_spread_proxy'] = np.abs(df['buy_qty'] - df['sell_qty']) / (df['volume'] + 1e-8)

    # New robust features, didn't include them when reaching 0.1218
    df['log_volume'] = np.log1p(df['volume'])
    df['bid_ask_imbalance'] = (df['bid_qty'] - df['ask_qty']) / (df['bid_qty'] + df['ask_qty'] + 1e-8)
    df['order_flow_imbalance'] = (df['buy_qty'] - df['sell_qty']) / (df['buy_qty'] + df['sell_qty'] + 1e-8)
    df['liquidity_ratio'] = (df['bid_qty'] + df['ask_qty']) / (df['volume'] + 1e-8)

    # Random feature for robustness
    # df['rand'] = np.random.normal(loc=0, scale=1, size=len(df))  ## for testing feature importance
    # Replace inf and -inf with NaN, then fill NaN with 0
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(0)  ## old, reaching 0.1218
    # For each column, replace NaN with median for robustness
    for col in df.columns:
        if df[col].isna().any():
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val if not pd.isna(median_val) else 0)
    return df 

engineered_cols = ["volume_weighted_sell", "buy_sell_ratio", "selling_pressure",\
                    "order_flow_imbalance", "log_volume", "liquidity_ratio"\
                     "bid_ask_imbalance", "effective_spread_proxy"
                ]

<a id='Config'></a>
# Configuration

In [53]:
class Config:
    TRAIN_PATH = "/Users/Sigrid/Desktop/drw-crypto-market-prediction/train.parquet"
    TEST_PATH = "/Users/Sigrid/Desktop/drw-crypto-market-prediction/test.parquet"
    SUBMISSION_PATH = "/Users/Sigrid/Desktop/drw-crypto-market-prediction/sample_submission.csv"

    FEATURES = [
        "X863", "X856", "X598", "X862", "X385", "X852", "X603", "X860", "X674",
        "X415", "X345", "X855", "X174", "X302", "X178", "X168", "X612", "X888", "X421", "X333",
        "buy_qty", "sell_qty", "volume", "bid_qty", "ask_qty",   ## initial version: has no bid_qty, ask_qty
        # "X181", "X30", "X28", "X19", "X21", "X20", "X26", "X175", "X27", "X22",  ## new features
        "X137", "X169", "X173", "X179", "X197", "X198", "X40" ## new feautures + more features, _2 = above + this line, _3 = this line, this line only gives the best result so far
        # "X175", "X22", "X28", "X181" ## _4 = the above line + this line
    ]

    LABEL_COLUMN = "label"
    N_FOLDS = 3
    RANDOM_STATE = 42

XGB_PARAMS = {
    "tree_method": "hist",
    "device": "gpu",
    "colsample_bylevel": 0.4778,
    "colsample_bynode": 0.3628,
    "colsample_bytree": 0.7107,
    "gamma": 1.7095,
    "learning_rate": 0.02213,
    "max_depth": 20,
    "max_leaves": 12,
    "min_child_weight": 16,
    "n_estimators": 1667,
    "subsample": 0.06567,
    "reg_alpha": 39.3524,
    "reg_lambda": 75.4484,
    "verbosity": 0,
    "random_state": Config.RANDOM_STATE,
    "n_jobs": -1
}

LGB_PARAMS = {
    # "boosting_type": "gbdt",
    # "objective": "regression",      
    # "metric": "mae",                
    # "colsample_bytree": 0.55,
    # "learning_rate": 0.021,
    # "min_child_samples": 32,
    # "min_child_weight": 0.15,
    # 'max_depth':-1,
    # "n_jobs": -1,
    # "num_leaves":64,
    # "random_state": 42,
    # "reg_alpha": 80,
    # "reg_lambda": 100,
    # "subsample": 0.85,
    # "verbosity": 1
    
    "n_estimators": 500,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "min_child_samples": 50,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 10,
    "reg_lambda": 10,
    "random_state": Config.RANDOM_STATE,
    "verbosity": -1,
    "n_jobs": -1
}

CAT_PARAMS = {
    "iterations":1000, 
    "learning_rate":0.03, 
    "loss_function":'MAE', 
    "verbose":False
}

QRF_PARAMS = {}

LEARNERS = [
    {"name": "xgb", "Estimator": XGBRegressor, "params": XGB_PARAMS}  ## initial version only has this learner (XGB) -> 0.1217, 0.1234
    # {"name": "lgb", "Estimator": LGBMRegressor, "params": LGB_PARAMS}   ## 0.11439
    # {"name": "cat", "Estimator": CatBoostRegressor, "params": CAT_PARAMS}
    # {"name": 'rfq', "Estimator": RandomForestQuantileRegressor, "params": QRF_PARAMS}
]

<a id='load'></a>
# Loading Data

In [54]:
def create_time_decay_weights(n: int, decay: float = 0.9) -> np.ndarray:
    positions = np.arange(n)
    normalized = positions / (n - 1)
    weights = decay ** (1.0 - normalized)
    return weights * n / weights.sum()

def load_data():
    train_df = pd.read_parquet(Config.TRAIN_PATH, columns=Config.FEATURES + [Config.LABEL_COLUMN])
    test_df = pd.read_parquet(Config.TEST_PATH, columns=Config.FEATURES)
    submission_df = pd.read_csv(Config.SUBMISSION_PATH)

    train_df = feature_engineering(train_df)
    test_df = feature_engineering(test_df)
    print(f"Loaded data - Train: {train_df.shape}, Test: {test_df.shape}, Submission: {submission_df.shape}")
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), submission_df

<a id='Train'></a>
# Training and Evaluation

In [ ]:
def get_model_slices(n_samples: int):
    return [
        {"name": "full_data", "cutoff": 0},
        {"name": "last_75pct", "cutoff": int(0.25 * n_samples)},
        {"name": "last_50pct", "cutoff": int(0.50 * n_samples)},
    ]

def train_and_evaluate(train_df, test_df):
    n_samples = len(train_df)
    model_slices = get_model_slices(n_samples)

    oof_preds = {
        learner["name"]: {s["name"]: np.zeros(n_samples) for s in model_slices}
        for learner in LEARNERS
    }
    test_preds = {
        learner["name"]: {s["name"]: np.zeros(len(test_df)) for s in model_slices}
        for learner in LEARNERS
    }


    full_weights = create_time_decay_weights(n_samples)
    kf = KFold(n_splits=Config.N_FOLDS, shuffle=False)

    for fold, (train_idx, valid_idx) in enumerate(kf.split(train_df), start=1):
        print(f"\n--- Fold {fold}/{Config.N_FOLDS} ---")
        X_valid = train_df.iloc[valid_idx][list(set(Config.FEATURES) | set(engineered_cols))]
        y_valid = train_df.iloc[valid_idx][Config.LABEL_COLUMN]

        for s in model_slices:
            cutoff = s["cutoff"]
            slice_name = s["name"]
            subset = train_df.iloc[cutoff:].reset_index(drop=True)
            rel_idx = train_idx[train_idx >= cutoff] - cutoff

            X_train = subset.iloc[rel_idx][list(set(Config.FEATURES) | set(engineered_cols))]
            y_train = subset.iloc[rel_idx][Config.LABEL_COLUMN]
            sw = create_time_decay_weights(len(subset))[rel_idx] if cutoff > 0 else full_weights[train_idx]

            print(f"  Training slice: {slice_name}, samples: {len(X_train)}")

            for learner in LEARNERS:
                model = learner["Estimator"](**learner["params"])
                if learner['name'] == 'rfq':
                    model.fit(X_train, y_train, sample_weight=sw)
                else:
                    model.fit(X_train, y_train, sample_weight=sw, eval_set=[(X_valid, y_valid)], verbose=200)

                mask = valid_idx >= cutoff
                if mask.any():
                    idxs = valid_idx[mask]
                    if learner['name'] == 'rfq':
                        oof_preds[learner["name"]][slice_name][idxs] = model.predict(train_df.iloc[idxs][list(set(Config.FEATURES)|set(engineered_cols))], quantiles=[0.5])
                    else:
                        oof_preds[learner["name"]][slice_name][idxs] = model.predict(train_df.iloc[idxs][list(set(Config.FEATURES)|set(engineered_cols))])
                if cutoff > 0 and (~mask).any():
                    oof_preds[learner["name"]][slice_name][valid_idx[~mask]] = oof_preds[learner["name"]]["full_data"][valid_idx[~mask]]

                if learner['name'] == 'rfq':
                    test_preds[learner["name"]][slice_name] += model.predict(test_df[list(set(Config.FEATURES) | set(engineered_cols))], quantiles = [0.5])
                else:
                    test_preds[learner["name"]][slice_name] += model.predict(test_df[list(set(Config.FEATURES) | set(engineered_cols))])

                # """
                # shap values graph 
                # """
                # explainer = shap.TreeExplainer(model, feature_perturbation="tree_path_dependent", model_output="raw")
                # shap_values = explainer.shap_values(X_valid[list(set(Config.FEATURES) | set(engineered_cols))])
                # # shap.summary_plot(shap_values, X_valid[list(set(Config.FEATURES) | set(engineered_cols))])
                # shap_df = np.abs(pd.DataFrame(shap_values, columns=list(set(Config.FEATURES) | set(engineered_cols)))).mean()
                # shap_df.sort_values(ascending=False, inplace=True)
                # print('Shap values for validation set:')
                # print(shap_df.tail())

    # Normalize test predictions
    for learner_name in test_preds:
        for slice_name in test_preds[learner_name]:
            test_preds[learner_name][slice_name] /= Config.N_FOLDS

    return oof_preds, test_preds, model_slices

<a id='Subm'></a>
# Submission

In [56]:
def ensemble_and_submit(train_df, oof_preds, test_preds, submission_df):
    learner_ensembles = {}
    oof_preds_full = {}
    for k, v in oof_preds.items():
        oof_preds_full[k] = {'full_data': v['full_data']}
    for learner_name in oof_preds_full:
        scores = {s: pearsonr(train_df[Config.LABEL_COLUMN], oof_preds_full[learner_name][s])[0]
                  for s in oof_preds_full[learner_name]}
        total_score = sum(scores.values())

        oof_simple = np.mean(list(oof_preds_full[learner_name].values()), axis=0)
        test_simple = np.mean(list(test_preds[learner_name].values()), axis=0)
        score_simple = pearsonr(train_df[Config.LABEL_COLUMN], oof_simple)[0]

        oof_weighted = sum(scores[s] / total_score * oof_preds_full[learner_name][s] for s in scores)
        test_weighted = sum(scores[s] / total_score * test_preds[learner_name][s] for s in scores)
        score_weighted = pearsonr(train_df[Config.LABEL_COLUMN], oof_weighted)[0]

        print(f"\n{learner_name.upper()} Simple Ensemble Pearson:   {score_simple:.4f}")
        print(f"{learner_name.upper()} Weighted Ensemble Pearson: {score_weighted:.4f}")

        learner_ensembles[learner_name] = {
            "oof_simple": oof_simple,
            "test_simple": test_simple,
            "oof_weighted": oof_weighted,
            "test_weighted": test_weighted
        }

    final_oof = np.mean([le["oof_weighted"] for le in learner_ensembles.values()], axis=0)
    final_test = np.mean([le["test_weighted"] for le in learner_ensembles.values()], axis=0)
    final_score = pearsonr(train_df[Config.LABEL_COLUMN], final_oof)[0]

    print(f"\nFINAL ensemble across learners Pearson: {final_score:.4f}")

    submission_df["prediction"] = final_test
    submission_df.to_csv("submission_XGB_weighted_0709_reduced.csv", index=False)
    print("Saved: submission_XGB_weighted_0709_reduced.csv")

<a id='Main'></a>
# Main 

In [57]:
if __name__ == "__main__":
    train_df, test_df, submission_df = load_data()
    oof_preds, test_preds, model_slices = train_and_evaluate(train_df, test_df)
    ensemble_and_submit(train_df, oof_preds, test_preds, submission_df)

Loaded data - Train: (525887, 39), Test: (538150, 38), Submission: (538150, 2)

--- Fold 1/3 ---
  Training slice: full_data, samples: 350591
[0]	validation_0-rmse:1.00228
[200]	validation_0-rmse:0.99395
[400]	validation_0-rmse:0.99501
[600]	validation_0-rmse:0.99783
[800]	validation_0-rmse:1.00223
[1000]	validation_0-rmse:1.00582
[1200]	validation_0-rmse:1.01022
[1400]	validation_0-rmse:1.01391
[1600]	validation_0-rmse:1.01889
[1666]	validation_0-rmse:1.01988
Shap values for validation set:
log_volume              0.004276
order_flow_imbalance    0.003539
bid_qty                 0.002470
buy_sell_ratio          0.002158
selling_pressure        0.001657
dtype: float32
  Training slice: last_75pct, samples: 350591
[0]	validation_0-rmse:1.00229
[200]	validation_0-rmse:0.99419
[400]	validation_0-rmse:0.99572
[600]	validation_0-rmse:0.99945
[800]	validation_0-rmse:1.00418
[1000]	validation_0-rmse:1.00813
[1200]	validation_0-rmse:1.01265
[1400]	validation_0-rmse:1.01688
[1600]	validation_0-

![Upvote](https://media.giphy.com/media/xT5LMHxhOfscxPfIfm/giphy.gif)

